# TINTO época 5 frente a L0: evaluación exploratoria en 2022

Solo inferencia: ninguno de los modelos se reentrena. TINTO es la variante PCA
con asignación única 32 × 32; L0 es el modelo de selección d50 guardado.

Antes de ejecutar: Entorno de ejecución → Cambiar tipo de entorno → acelerador
**Ninguno (CPU)**. Ejecutar las celdas en orden. Este notebook no se ha ejecutado
aún en Colab; la prueba local de flujo usa dos días y no produce métricas.

Necesitas el paquete `tinto_evaluacion_2022_20261029.tar.gz` preparado en el repo.
No hace falta subir los 2.8 GB de caché de entrenamiento. Se usarán 2020–2022
para perfiles causales y normalización heredada; se puntúa únicamente 2022.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
archives = [Path(n) for n in uploaded if n.endswith('.tar.gz')]
assert len(archives) == 1, 'Sube solo el paquete de evaluación tar.gz.'
archive = archives[0]


In [ ]:
import tarfile, json, hashlib
ROOT = Path('/content/tinto_evaluacion')
ROOT.mkdir(exist_ok=True)
if not (ROOT/'MANIFIESTO.json').exists():
    with tarfile.open(archive) as t:
        t.extractall(ROOT, filter='data')
manifest = json.loads((ROOT/'MANIFIESTO.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
print('Paquete verificado:', len(manifest), 'archivos')


## Entorno fijado

Se instala Python 3.14.7 y el lock del repositorio en un entorno aparte.
La instalación requiere red y puede tardar. Si una versión no está disponible,
detener y revisar; no sustituir versiones para continuar silenciosamente.
No se instala TINTOlib: el mapa y escalador ya están ajustados y guardados.


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.14.7'], check=True)
VENV = Path('/content/tinto_eval_env')
if not VENV.exists():
    subprocess.run(['uv', 'venv', '--python', '3.14.7', str(VENV)], check=True)
PY = str(VENV/'bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', str(ROOT/'requirements-lock.txt')], check=True)
ENV = os.environ.copy()
ENV.update(CUDA_VISIBLE_DEVICES='-1', TF_ENABLE_ONEDNN_OPTS='1', TF_NUM_INTRAOP_THREADS='2', TF_NUM_INTEROP_THREADS='1', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1', NUMEXPR_NUM_THREADS='1', MPLCONFIGDIR='/tmp/tinto-mpl')
subprocess.run([PY, '-c', "import sys,tensorflow as tf,keras; print(sys.version, tf.__version__, keras.__version__); assert not tf.config.get_visible_devices('GPU')"], env=ENV, check=True)


## Prueba breve

Comprueba carga, conversión y control de L0 con sus puntajes guardados. No calcula
precisión, recall ni F1. Si falla el control, detenerse antes de interpretar resultados.


In [ ]:
SCRIPT = str(ROOT/'experiments/exploracion_tinto/evaluar_2022.py')
SAL = ROOT/'results/exploracion_tinto_2022/evaluacion_colab_20261029'
TEST = SAL/'prueba'
if not (TEST/'PRUEBA_OK.json').exists():
    subprocess.run([PY, SCRIPT, '--prueba', '--salida', str(TEST)], cwd=ROOT, env=ENV, check=True)
print((TEST/'PRUEBA_OK.json').read_text())


## Evaluación completa: dos procesos en paralelo, solo CPU

Cada proceso recorre 2022 limpio y diez calendarios para δ = 100, 50 y 20 %.
Ambos usan los mismos modelos congelados, entradas y reglas científicas. Se
limitan a dos hilos de cómputo TensorFlow por proceso; las bibliotecas BLAS no
abren un grupo adicional de muchos hilos. El tiempo dependerá de la CPU asignada.

La primera celda lanza ambas corridas y devuelve el control. La siguiente
escribe un estado y muestra avance cada cinco minutos hasta terminar. Los logs
y los puntajes diarios se guardan mientras corren. No hay reanudación parcial
automática ni se vuelven a lanzar procesos al repetir la celda por accidente.

Se comparan los mismos días con promedio diario y mínimo 12 horas válidas.
p99/p99.9 salen del mismo conjunto limpio 2022 por detector: son umbrales
exploratorios, no calibración final de 2023. No cambiar batch, arquitectura o
umbrales para buscar resultados favorables. Si falla la prueba breve, no lanzar.


In [ ]:
import time
assert (TEST/'PRUEBA_OK.json').exists(), 'Primero debe pasar la prueba breve.'
test_ok = json.loads((TEST/'PRUEBA_OK.json').read_text())
assert test_ok.get('solo_flujo') and test_ok.get('modelos_cargados') and test_ok.get('sin_metricas')
SAL.mkdir(parents=True, exist_ok=True)
# Reutilizar los handles si se repite esta celda, sin duplicar procesos.
if 'EVAL_PROCS' not in globals():
    EVAL_PROCS = {}
for name in ['corrida1', 'corrida2']:
    out = SAL/name
    if (out/'LISTO.json').exists():
        print(name, 'ya completa; se conserva')
        continue
    if name in EVAL_PROCS and EVAL_PROCS[name].poll() is None:
        print(name, 'ya está ejecutándose; no se duplica')
        continue
    assert not out.exists(), 'Carpeta parcial existente: conservar y revisar antes de repetir en otra carpeta.'
    log = SAL/f'{name}.log'
    with log.open('w') as fh:
        EVAL_PROCS[name] = subprocess.Popen(
            [PY, '-u', SCRIPT, '--salida', str(out), '--batch', '512', '--bootstrap', '2000'],
            cwd=ROOT, env=ENV, stdout=fh, stderr=subprocess.STDOUT)
    print(name, 'lanzada; PID', EVAL_PROCS[name].pid, 'log:', log)

def estado_evaluaciones():
    rows = []
    for name in ['corrida1', 'corrida2']:
        out = SAL/name
        log = SAL/f'{name}.log'
        text = log.read_text(errors='replace') if log.exists() else ''
        process = EVAL_PROCS.get(name)
        code = process.poll() if process is not None else None
        marker = (out/'LISTO.json').exists()
        complete = marker and (process is None or code == 0)
        failed = process is not None and code is not None and (code != 0 or not complete)
        completed = text.count(': puntajes diarios guardados')
        row = dict(corrida=name, pid=process.pid if process else None,
                   estado='fallida' if failed else 'completa' if complete else 'en_curso',
                   matrices_con_puntajes=completed, matrices_esperadas=31,
                   codigo_salida=code)
        if failed:
            row['ultimas_lineas_log'] = text[-2000:]
        rows.append(row)
    report = dict(actualizado_UTC=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                  corridas=rows, terminadas=sum(r['estado']=='completa' for r in rows))
    # Reemplazo atómico para que el monitor lea siempre JSON completo.
    temp = SAL/'ESTADO_EVALUACION.tmp'
    temp.write_text(json.dumps(report, indent=2, ensure_ascii=False))
    temp.replace(SAL/'ESTADO_EVALUACION.json')
    return report

print(json.dumps(estado_evaluaciones(), indent=2, ensure_ascii=False))


In [ ]:
# Comprobar cada cinco minutos sin relanzar evaluaciones.
while True:
    state = estado_evaluaciones()
    print(json.dumps(state, indent=2, ensure_ascii=False), flush=True)
    failed = [r for r in state['corridas'] if r['estado']=='fallida']
    if failed:
        raise RuntimeError('Falló alguna corrida; revisar logs y descargar avances. No se relanza automáticamente.')
    if state['terminadas'] == 2:
        break
    time.sleep(300)


In [ ]:
import pandas as pd
for name in ['resultados.csv', 'metricas_por_calendario.csv', 'intervalos_exploratorios.csv', 'dias_comunes.csv']:
    pd.testing.assert_frame_equal(pd.read_csv(SAL/'corrida1'/name), pd.read_csv(SAL/'corrida2'/name), check_exact=True)
for name in ['umbrales.json', 'control_L0.json']:
    assert json.loads((SAL/'corrida1'/name).read_text()) == json.loads((SAL/'corrida2'/name).read_text())
for p in (SAL/'corrida1').glob('*_dias.csv.gz'):
    pd.testing.assert_frame_equal(pd.read_csv(p), pd.read_csv(SAL/'corrida2'/p.name), check_exact=True)
(SAL/'REPRODUCIBLE.json').write_text(json.dumps({'dos_evaluaciones_identicas': True}, indent=2))
r = pd.read_csv(SAL/'corrida1/resultados.csv')
display(r[r.version=='con_ataque'][['delta_pct','detector','punto','precision','recall','f1','tp','fp','fn']])


## Descargar resultados o avances

Las métricas corresponden a validación ya reutilizada y a un modelo cuyo entrenamiento
se cerró manualmente: no son una prueba independiente. Bootstrap por estación-mes,
calendarios juntos; no cubre incertidumbre de entrenamiento ni umbral. No se vuelven
a elegir modelos con estas métricas. Descarga el ZIP para revisarlo en el chat.

Si hay un fallo, también puedes ejecutar la última celda para guardar logs y avances,
sin volver a lanzar la evaluación. Un ZIP de avance no implica una evaluación terminada.


In [ ]:
import shutil
zip_path = shutil.make_archive('/content/tinto_evaluacion_2022_resultados', 'zip', root_dir=SAL)
files.download(zip_path)
